# GuardNet — 04. Inference

Loads the trained model + preprocessor and scores **real NSL-KDD records**,
producing genuine attack probabilities. No random inputs, no label leakage —
every prediction comes from the model. Mirrors `scripts/threat_detector.py`.

In [ ]:
import os, sys
import numpy as np
import pandas as pd

BASE = os.path.dirname(os.getcwd())
sys.path.append(os.path.join(BASE, 'scripts'))
from threat_detector import load_artifacts, preprocess, COLUMNS

model, scaler, feature_columns, threshold = load_artifacts()
print('Decision threshold =', threshold)

In [ ]:
# Take a random sample of real test records and score them
TEST_PATH = os.path.join(BASE, 'data', 'NSL-KDD Dataset', 'KDDTest+.txt')
df = pd.read_csv(TEST_PATH, header=None, names=COLUMNS).sample(15, random_state=0).reset_index(drop=True)

X = preprocess(df, scaler, feature_columns)
probs = model.predict(X, verbose=0).flatten()

out = pd.DataFrame({
    'protocol': df['protocol_type'],
    'service': df['service'],
    'threat_score_%': (probs * 100).round(1),
    'prediction': np.where(probs >= threshold, 'ATTACK', 'BENIGN'),
    'actual': np.where(df['label'] == 'normal', 'BENIGN', 'ATTACK'),
})
out

In [ ]:
correct = (out['prediction'] == out['actual']).sum()
print(f'Correct on this sample: {correct}/{len(out)}')
print(f'Threats flagged: {(out.prediction == "ATTACK").sum()}')

The same model powers the live Flask dashboard (`python app.py`), which streams
real NSL-KDD records through it and visualizes predictions in real time.